## 消息（Messages）

在调用模型时，发送给LLM的消息、LLM返回的消息都包含以下几部分内容：
- `role`：消息所属角色，可以是system、user、assistant、tool
- `content`：消息的内容
- `metadata`（可选）：消息的元数据，例如：消息的ID、消耗的token等

### 1. 消息类型

在LangChain中，我们并不需要自己创建`BaseMessag`e对象，LangChain已经把常见消息根据角色（Role）创建了对应的`BaseMessage`的子类：
- `SystemMessage`：role是system，代表系统消息，用于设定模型角色和交互背景
- `HumanMessage`：role是user，代表用户输入的消息
- `AIMessage`：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- `ToolMessage`：role是tool，代表工具调用时产生的结果

### 2. Agent 调用消息

之前我们都是自己用dict模拟消息：
```
response = agent.invoke({
    "messages": [
        {"role": "system", "content": "你是智能问答助手"},
        {"role": "user", "content": "你好，我是虎哥"},
        {"role": "assistant", "content": "你好，虎哥，很高兴认识你。"},
        {"role": "user", "content": "我的名字是什么？"},
    ]
})

```

现在我们可以这样传递消息列表：

In [24]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

# 定义工具并调用
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print('tool调用：get_weather')
    return f'{location} 城市， 今天是晴天！'

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage(content="你是智能问答助手"),
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？, 今天昆山天气如何？")
    ]
})

for message in response['messages']:
    message.pretty_print()

tool调用：get_weather
================================ System Message ================================

你是智能问答助手
================================ Human Message =================================

你好，我是虎哥
================================== Ai Message ==================================

你好，虎哥，很高兴认识你。
================================ Human Message =================================

我的名字是什么？, 今天昆山天气如何？
================================== Ai Message ==================================

虎哥，你的名字是虎哥。我帮你查一下今天昆山的天气。
Tool Calls:
  get_weather (call_00_xAMhvJa4JofEv6DACjag7880)
 Call ID: call_00_xAMhvJa4JofEv6DACjag7880
  Args:
    location: 昆山
================================= Tool Message =================================
Name: get_weather

昆山 城市， 今天是晴天！
================================== Ai Message ==================================

虎哥，今天昆山是晴天。


### 3. LLM 直接调用消息



工具消息

In [26]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="deepseek-flash")

# 定义工具并调用
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print('tool调用：get_weather')
    return f'{location} 城市， 今天是晴天！'

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("长沙天气怎么样?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")  # Tool: get_weather 工具名（即函数名）
    print(f"Args: {tool_call['args']}")  # Args: {'location': '长沙'} 传给工具的参数
    print(f"ID: {tool_call['id']}")  # ID: call_e68e9d4bd0e64d8a8c54ba 唯一标识符，用于后续匹配工具执行结果
print('-' * 100)
print(response)

Tool: get_weather
Args: {'location': '长沙'}
ID: call_00_x9SK4dGiKp4aFzJrN1eD4492
----------------------------------------------------------------------------------------------------
content='' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer needs weather. Call tool.'} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 293, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 9, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 165}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '460be66f-ced7-4bb6-b062-675444ad03ca', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a19b49-c0d7-7128-9006-d2c699868b01-0' tool_calls=[{'name': 'get_weather', 'args': {'locati

两种写法等价：字典格式 ↔ Message 对象（推荐用对象，可读性更好、功能更全）

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

model = init_chat_model(model="deepseek-flash")

# 写法一：字典格式
conversation_dict = [
    {"role": "system", "content": "你是翻译助手"},
    {"role": "user", "content": "翻译：我喜欢编程"},
]

# 写法二：Message 对象（推荐）
conversation_obj = [
    SystemMessage(content="你是翻译助手"),
    HumanMessage(content="翻译：我喜欢编程"),
]

# 两种写法效果完全一样
print("字典格式：", model.invoke(conversation_dict).content)
print("对象格式：", model.invoke(conversation_obj).content)

消息内部结构

In [1]:
# 一条 AIMessage 不止有 content，还携带很多元信息。用 model_dump_json() 可以看到完整字段。
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-flash")
response = model.invoke("你好")

# ⭐ 查看 AIMessage 的完整字段
print(response.model_dump_json(indent=2))

{
  "content": "你好！很高兴见到你。有什么我可以帮你的吗？😊",
  "additional_kwargs": {
    "refusal": null,
    "reasoning_content": "The user just said \"你好\" (hello) in Chinese. I should respond in Chinese with a friendly greeting and offer to help."
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 42,
      "prompt_tokens": 31,
      "total_tokens": 73,
      "completion_tokens_details": {
        "accepted_prediction_tokens": null,
        "audio_tokens": null,
        "reasoning_tokens": 27,
        "rejected_prediction_tokens": null,
        "text_tokens": null
      },
      "prompt_tokens_details": {
        "audio_tokens": null,
        "cache_write_tokens": null,
        "cached_tokens": 0,
        "image_tokens": null,
        "text_tokens": null
      },
      "prompt_cache_hit_tokens": 0,
      "prompt_cache_miss_tokens": 31
    },
    "model_provider": "deepseek",
    "model_name": "deepseek-flash",
    "system_fingerprint": "aeb56401ca74e127821c4f9126dcb669",
   

```
AIMessage
├── content              最终文本
├── additional_kwargs    附加信息（如推理内容 reasoning_content）
├── response_metadata    模型原始元数据（token 用量、模型名）
├── id                   消息唯一 ID
├── tool_calls           工具调用（Agent 阶段用）
└── usage_metadata       标准化 token 用量
```

In [2]:
# ⭐ 逐字段访问
print("正文：", response.content)
print("推理内容：", response.additional_kwargs.get("reasoning_content"))
print("token 用量：", response.usage_metadata)
print("模型元数据：", response.response_metadata)

正文： 你好！很高兴见到你。有什么我可以帮你的吗？😊
推理内容： The user just said "你好" (hello) in Chinese. I should respond in Chinese with a friendly greeting and offer to help.
token 用量： {'input_tokens': 31, 'output_tokens': 42, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 27}}
模型元数据： {'token_usage': {'completion_tokens': 42, 'prompt_tokens': 31, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '9c85094f-a0d5-4fa2-9ede-ccbe3e102e16', 'finish_reason': 'stop', 'logprobs': None}
